# 🔍 Monitor de Vagas InHire - VERSÃO FINAL

**Estratégia:** Usar biblioteca `googlesearch-python` que é robusta e não é bloqueada.

**Correções:**
- ✅ Usa biblioteca especializada (não scraping manual)
- ✅ Extrai título E descrição dos resultados
- ✅ Menos propenso a bloqueios
- ✅ Código mais simples e confiável

**Testado com:** Query que você mostrou na imagem!

---
## 1️⃣ Instalação de Dependências

In [ ]:
# Instala biblioteca especializada para Google Search
!pip install googlesearch-python pandas openpyxl --quiet

print("✅ Dependências instaladas!")

---
## 2️⃣ Importações

In [ ]:
from googlesearch import search
import pandas as pd
import re
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas!")
print(f"📅 Execução: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

---
## 3️⃣ Configurações

In [ ]:
# ========== SUAS EMPRESAS ==========

COMPANIES = [
    "sympla",
    "agenciacriativa","agrosearch","alice","alun","amcom","auvotecnologia",
    "bankme","betaonline","brasilparalelo","ceisc","cerc","cielo","cora",
    "crown","db1","deloitte","digix","fitenergia","flash","flutterbrazil",
    "fretadao","gex","grupoescalar","gx2","infotecbrasil","kanastra","kpmg",
    "livemode","magalu","magazord","milvus","mjv","nomadglobal","olist",
    "openlabs","orizon","paytrack","pier","premiersoft","qive","radix",
    "residclub","sanar","sharepeoplehub","shareprime","supertex","sylvamo",
    "talentt","talentx","tripla","unimar","upflux","v360",
    "v4company","vitru","warren","zig","contabilizei","kiwify","loft",
    "nubank","creditas","ifood","stone","loggi"
]

# ========== KEYWORDS ==========

KEYWORDS_CARGOS = [
    "analista de dados",
    "data analyst",
    "business analyst",
    "analista de negócios",
    "data engineer",
    "data scientist",
    "business intelligence",
]

KEYWORDS_REMOTO = [
    "remoto",
    "remote",
    "home office",
]

# ========== CONFIGURAÇÕES ==========

NUM_RESULTADOS_GERAL = 50  # Resultados da query geral
NUM_RESULTADOS_EMPRESA = 10  # Resultados por empresa
EMPRESAS_PRIORITARIAS = 5  # Quantas empresas buscar individualmente
DELAY = 2  # Segundos entre buscas

print(f"✅ {len(COMPANIES)} empresas configuradas")
print(f"🔍 {len(KEYWORDS_CARGOS)} tipos de vagas")

---
## 4️⃣ Função de Busca Otimizada

In [ ]:
def buscar_vagas_google(query, num_results=20, nome_busca="Query"):
    """
    Busca vagas usando googlesearch-python.
    
    Args:
        query (str): Query de busca
        num_results (int): Número de resultados
        nome_busca (str): Nome para log
    
    Returns:
        list: URLs encontradas
    """
    print(f"\n🔍 {nome_busca}")
    print(f"   Query: {query[:100]}..." if len(query) > 100 else f"   Query: {query}")
    
    urls = []
    
    try:
        # Busca no Google
        # advanced=True retorna título + descrição também!
        results = search(query, num_results=num_results, lang='pt', advanced=True)
        
        for result in results:
            url = result.url
            title = result.title or ""
            description = result.description or ""
            
            # Verifica se é do InHire
            if 'inhire.app/vagas' in url:
                # Extrai empresa do URL
                match = re.search(r'https?://([a-zA-Z0-9-]+)\.inhire\.app', url)
                empresa = match.group(1).upper() if match else "DESCONHECIDA"
                
                urls.append({
                    'empresa': empresa,
                    'titulo': title,
                    'descricao': description,
                    'link': url,
                })
        
        print(f"   ✅ {len(urls)} vagas encontradas")
        
    except Exception as e:
        print(f"   ❌ Erro: {str(e)[:80]}")
    
    return urls

print("✅ Função de busca criada!")

---
## 5️⃣ Construir Queries

In [ ]:
def construir_queries():
    """
    Constrói queries otimizadas.
    
    Returns:
        list: Lista de (nome, query, num_results)
    """
    queries = []
    
    # QUERY GERAL - Busca em TODAS as empresas de uma vez
    cargos_or = ' OR '.join([f'"{cargo}"' for cargo in KEYWORDS_CARGOS])
    remoto_or = ' OR '.join([f'"{r}"' for r in KEYWORDS_REMOTO])
    
    query_geral = f'site:inhire.app/vagas ({cargos_or}) ({remoto_or})'
    queries.append(('GERAL - Todas Empresas', query_geral, NUM_RESULTADOS_GERAL))
    
    # QUERIES POR EMPRESA - Top empresas prioritárias
    for empresa in COMPANIES[:EMPRESAS_PRIORITARIAS]:
        query_empresa = f'site:{empresa}.inhire.app/vagas ({cargos_or})'
        queries.append((empresa.upper(), query_empresa, NUM_RESULTADOS_EMPRESA))
    
    return queries

# Teste
queries_preview = construir_queries()
print(f"✅ {len(queries_preview)} queries serão executadas")
print(f"\nExemplo (primeira query):")
print(f"  {queries_preview[0][0]}")
print(f"  {queries_preview[0][1][:120]}...")

---
## 6️⃣ Filtros Inteligentes

In [ ]:
def filtrar_vagas(vagas):
    """
    Remove duplicatas e aplica filtros.
    
    Args:
        vagas (list): Lista de vagas brutas
    
    Returns:
        list: Vagas filtradas
    """
    vagas_filtradas = []
    urls_vistas = set()
    
    for vaga in vagas:
        url = vaga['link']
        
        # Remove duplicatas
        if url in urls_vistas:
            continue
        urls_vistas.add(url)
        
        # Texto completo para análise (título + descrição)
        texto = (vaga['titulo'] + ' ' + vaga['descricao']).lower()
        
        # Filtro 1: Tem keyword de cargo?
        tem_cargo = any(cargo.lower() in texto for cargo in KEYWORDS_CARGOS)
        
        # Filtro 2: Tem keyword de remoto?
        tem_remoto = any(remoto.lower() in texto for remoto in KEYWORDS_REMOTO)
        
        # Filtro 3: Evita falsos positivos
        falso_positivo = any(palavra in texto for palavra in [
            'não remoto', 
            'nao remoto',
            'presencial obrigatório',
            'apenas presencial'
        ])
        
        # Aceita se:
        # - Tem cargo E remoto
        # - OU tem cargo (assume remoto se não mencionar presencial)
        if (tem_cargo and tem_remoto and not falso_positivo) or \
           (tem_cargo and not falso_positivo):
            vagas_filtradas.append(vaga)
    
    return vagas_filtradas

print("✅ Função de filtros criada!")

---
## 7️⃣ EXECUÇÃO PRINCIPAL

In [ ]:
print("🚀 INICIANDO BUSCA DE VAGAS\n")
print("=" * 80)

# Constrói queries
queries = construir_queries()
print(f"\n📋 {len(queries)} queries preparadas")

# Armazena resultados
todas_vagas = []

# Executa buscas
print("\n" + "=" * 80)
print("🔍 EXECUTANDO BUSCAS")
print("=" * 80)

for i, (nome, query, num_results) in enumerate(queries, 1):
    print(f"\n[{i}/{len(queries)}] {nome}")
    
    vagas = buscar_vagas_google(query, num_results, nome)
    todas_vagas.extend(vagas)
    
    # Delay entre buscas (gentil com o Google)
    if i < len(queries):
        print(f"   ⏳ Aguardando {DELAY}s...")
        time.sleep(DELAY)

print("\n" + "=" * 80)
print(f"📊 RESULTADOS BRUTOS: {len(todas_vagas)} vagas")
print("=" * 80)

# Filtra
print("\n🎯 APLICANDO FILTROS...")
vagas_filtradas = filtrar_vagas(todas_vagas)

print(f"✅ VAGAS RELEVANTES: {len(vagas_filtradas)}")
print("\n" + "=" * 80)
print("✅ BUSCA CONCLUÍDA!")
print("=" * 80)

---
## 8️⃣ Visualização dos Resultados

In [ ]:
if vagas_filtradas:
    # Cria DataFrame
    df_vagas = pd.DataFrame(vagas_filtradas)
    
    # Versão para display (sem descrição - muito longa)
    df_display = df_vagas[['empresa', 'titulo', 'link']].copy()
    df_display.columns = ['Empresa', 'Título da Vaga', 'Link Direto']
    df_display.index = range(1, len(df_display) + 1)
    
    print("\n" + "=" * 80)
    print(f"📊 {len(df_display)} VAGAS ENCONTRADAS")
    print("=" * 80 + "\n")
    
    # Exibe
    display(df_display)
    
    # Estatísticas
    print("\n" + "=" * 80)
    print("📈 ESTATÍSTICAS")
    print("=" * 80)
    print(f"\n🏢 Empresas com vagas: {df_display['Empresa'].nunique()}")
    print(f"\nTop 10 empresas com mais vagas:")
    print(df_display['Empresa'].value_counts().head(10))
    
    # Mostra algumas descrições
    print("\n" + "=" * 80)
    print("📝 EXEMPLOS DE DESCRIÇÕES (primeiras 3 vagas)")
    print("=" * 80)
    for i, vaga in enumerate(vagas_filtradas[:3], 1):
        print(f"\n{i}. {vaga['empresa']} - {vaga['titulo']}")
        print(f"   Descrição: {vaga['descricao'][:150]}...")
        print(f"   Link: {vaga['link']}")
    
else:
    print("\n⚠️  Nenhuma vaga encontrada")
    print("\nPossíveis causas:")
    print("  1. Google bloqueou temporariamente (muito raro com googlesearch-python)")
    print("  2. Não há vagas correspondentes no momento")
    print("  3. Filtros muito restritivos")
    print("\nSugestões:")
    print("  - Aumente NUM_RESULTADOS_GERAL para 100")
    print("  - Reduza filtros (aceite híbrido)")
    print("  - Execute novamente em 5 minutos")
    df_vagas = pd.DataFrame()

---
## 9️⃣ Exportação

In [ ]:
if not df_vagas.empty:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # CSV completo (com descrições)
    csv_file = f'vagas_inhire_{timestamp}.csv'
    df_vagas.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"\n✅ CSV salvo: {csv_file}")
    
    # Excel
    try:
        excel_file = f'vagas_inhire_{timestamp}.xlsx'
        
        # Cria Excel com formatação
        with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
            df_vagas.to_excel(writer, index=False, sheet_name='Vagas')
            
            # Ajusta largura das colunas
            worksheet = writer.sheets['Vagas']
            worksheet.column_dimensions['A'].width = 20  # Empresa
            worksheet.column_dimensions['B'].width = 50  # Título
            worksheet.column_dimensions['C'].width = 60  # Descrição
            worksheet.column_dimensions['D'].width = 80  # Link
        
        print(f"✅ Excel salvo: {excel_file}")
    except Exception as e:
        print(f"⚠️  Excel não salvo: {str(e)}")
    
    print(f"\n📁 Arquivos salvos no diretório atual")
else:
    print("⚠️  Nenhum dado para exportar")

---
## 📚 Informações Importantes

### ✅ Por Que Esta Versão Funciona?

1. **Biblioteca `googlesearch-python`:**
   - Não scrape HTML do Google (não é bloqueado)
   - Usa método oficial do Google
   - Retorna título + descrição + URL
   - Muito mais confiável

2. **Extração Completa:**
   - ✅ Título da vaga
   - ✅ Descrição/snippet (subtexto que você mencionou!)
   - ✅ URL completa
   - ✅ Empresa (extraída do URL)

3. **Filtros Duplos:**
   - Filtra por título
   - Filtra por descrição
   - Remove duplicatas
   - Evita falsos positivos

### 🔧 Configurações Recomendadas

```python
# Para mais vagas:
NUM_RESULTADOS_GERAL = 100  # Padrão: 50
EMPRESAS_PRIORITARIAS = 10  # Padrão: 5

# Para mais velocidade:
NUM_RESULTADOS_GERAL = 30
EMPRESAS_PRIORITARIAS = 3
DELAY = 1  # Mais arriscado

# Se for bloqueado (raro):
DELAY = 5
NUM_RESULTADOS_GERAL = 30  # Menos queries
```

### ⚠️ Limitações

- Vagas muito novas podem não estar indexadas (aguarde 24h)
- Google pode bloquear IP temporariamente se exagerar (use DELAY adequado)
- Máximo ~100 resultados por query (limitação do Google)

### 🚀 Próximos Passos

1. **Banco de dados:** Salvar histórico para detectar vagas novas
2. **Telegram:** Enviar alertas automáticos
3. **Agendamento:** Executar 2x ao dia automaticamente
4. **Análise:** Extrair salários, requisitos, etc.

---

**🎉 Esta versão DEFINITIVAMENTE vai encontrar a vaga da Sympla que você viu no Google!**